# Week 12 Live Coding: Where Should the Last \$50 Million Go?

Seven states, 93 electoral votes, and you need 44 of them. Today we:

1. Simulate 200,000 elections and read your chance of reaching 270 off the picture.
2. Let each state's polls miss independently, and watch your chance drop.
3. See where the analytics director's 0.07 came from.
4. Build diminishing returns, and run the first round of the algorithm by hand.
5. Run all fifty rounds, once for each goal.
6. Rerun the plan at the top and bottom of her interval.

### Before you start

**There is no problem set this week.** Before you leave, print this notebook to PDF and upload it to the **in-class** assignment on Canvas (the steps are at the bottom). That is your attendance credit. It is not graded on being right.

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

sw = pd.read_csv(
    'https://raw.githubusercontent.com/joshuakalla/'
    'data_science_campaigns_26/main/weeks/'
    'wk12_presidential_allocation/data/swing_states.csv')
ad = pd.read_csv(
    'https://raw.githubusercontent.com/joshuakalla/'
    'data_science_campaigns_26/main/weeks/'
    'wk12_presidential_allocation/data/ad_spending_effects.csv')

BASE = 226                         # electoral votes you already hold
EV = sw['electoral_votes'].values  # .values: the column as a plain array
m0 = sw['margin'].values           # your poll margin in each state
sw

## Part 1: Simulate the election

You win or lose each state, and then you count electoral votes.

Polls miss in two ways at once. A **national** miss hits every state in the same direction (in 2020 the polls overstated Biden almost everywhere). A **state** miss is different in each state. From Week 8, the national part is the bigger one.

In [ ]:
rng = np.random.default_rng(2026)   # a random number generator, seed 2026
N = 200000                          # number of simulated elections

# One national miss per election, sd 3.5 points. Shape (N, 1) is one
# column, so the same miss is added to all seven states.
national = rng.normal(0, 3.5, (N, 1))

# A different state miss for each state in each election, sd 2.0 points
state = rng.normal(0, 2.0, (N, 7))

# one row per election, one column per state
result = m0 + national + state
won = result > 0                     # True where you carried the state
evs = BASE + (won * EV).sum(axis=1)  # axis=1: add across each row

print('average electoral votes:', round(evs.mean(), 1))
print('chance of reaching 270:', round(100 * (evs >= 270).mean(), 1), '%')

*Check: 261.8 and 39.5%.*

You never get 261.8 electoral votes. Here is what you do get.

In [ ]:
# share of the 200,000 elections that ended at each electoral-vote total
counts = pd.Series(evs).value_counts(normalize=True).sort_index()
wins = counts[counts.index >= 270]

plt.bar(counts.index, counts.values, width=4, color='gray')
plt.bar(wins.index, wins.values, width=4, color='#0F4D92')  # reached 270
plt.axvline(269.5, color='#B58900')                          # the 270 line
plt.xlabel('Your electoral votes')
plt.ylabel('Share of simulations')
plt.show()

The two tall bars are losing all seven states (226) and winning all seven (319). That is the national miss: when the polls are wrong, they are wrong in every state at once.

## Part 2: What if states missed independently?

Give each state the same total uncertainty, 4.03 points, but let every state miss on its own.

In [ ]:
total_sd = np.sqrt(3.5**2 + 2.0**2)   # 4.03, from the slides
print('total sd in one state:', round(total_sd, 2))

# its own generator, so re-running this cell gives the same answer
indep = np.random.default_rng(33).normal(0, total_sd, (N, 7))
evs_indep = BASE + (((m0 + indep) > 0) * EV).sum(axis=1)

print('misses move together:', round(100 * (evs >= 270).mean(), 1), '%')
print('misses independent:', round(100 * (evs_indep >= 270).mean(), 1), '%')

*Check: 4.03, then 39.5% and 33.4%.*

You are behind in five of the seven states. With independent misses you need five separate misses in your favor. With a shared miss, one national miss in your favor helps you in all five at once.

## Part 3: Where the analytics director's 0.07 came from

Sixty past campaigns. For each one: how much more (or less) it spent on ads in a state than its opponent, and how much its margin in that state moved. Same regression syntax as earlier weeks.

In [ ]:
reg = smf.ols('margin_shift_pp ~ spend_advantage_m', data=ad).fit()
coef = reg.params['spend_advantage_m']

# conf_int() gives the 95% interval: [0] is the low end, [1] the high end
ci = reg.conf_int().loc['spend_advantage_m']
lo = ci[0]
hi = ci[1]

print('$1M moves a state', round(coef, 3), 'points')
print('95% interval:', round(lo, 3), 'to', round(hi, 3))

*Check: 0.071, and 0.033 to 0.109.*

**Nobody randomized any of this.** Campaigns choose where to spend. If they put more money where they were already gaining, this line credits the ads for gains they did not cause. The interval does not fix that: like a poll's margin of error, it covers sampling error only.

We build the plan on 0.07 anyway. Part 6 asks what happens if it is wrong.

## Part 4: Diminishing returns, and the first million by hand

We assume no amount of money moves one state more than 40 × 0.07 = 2.8 points (we made this up). The formula below gets there slowly: each million buys a little less than the one before.

In [ ]:
# np.exp(x) is e to the power x. As d grows, np.exp(-d / 40) shrinks
# toward 0, so the margin bought rises toward 40 * coef and stops there.
for d in [1, 10, 25, 50]:
    bought = 40 * coef * (1 - np.exp(-d / 40))
    print('$' + str(d) + 'M buys', round(bought, 2), 'points.',
          'A straight line says', round(coef * d, 2))

*Check: \$50M buys 2.02 points, against 3.54 on a straight line.*

Now round 1 of the algorithm. For each state: put \$1M there and nowhere else, rerun the same 200,000 elections, and see how much your chance of reaching 270 goes up.

In [ ]:
before = (evs >= 270).mean()      # 39.5%, nothing spent yet

for i in range(7):
    spend = np.zeros(7)           # $M in each state
    spend[i] = 1                  # $1M in state i, nothing anywhere else
    margins = m0 + 40 * coef * (1 - np.exp(-spend / 40))
    won_new = (margins + national + state) > 0
    evs_new = BASE + (won_new * EV).sum(axis=1)
    gain = 100 * ((evs_new >= 270).mean() - before)
    print(sw['state'][i], EV[i], 'EVs, margin', m0[i],
          '-> the next $1M buys', round(gain, 3), 'points')

*Check: Pennsylvania +0.267, Nevada +0.039.*

Pennsylvania comes first: big and close. Nevada is closer than Georgia, Arizona and North Carolina and comes last, because six electoral votes rarely decide the election.

## Part 5: The allocator, with two goals

Fifty rounds. Each round does what the cell above did, then gives the million to the state where it raised your chance of 270 the most.

In [ ]:
plan_270 = np.zeros(7)            # $M given to each state so far

for r in range(50):
    scores = []
    for i in range(7):
        spend = plan_270.copy()   # .copy(): leave plan_270 as it is
        spend[i] = spend[i] + 1   # one more $1M in state i
        margins = m0 + 40 * coef * (1 - np.exp(-spend / 40))
        won_new = (margins + national + state) > 0
        evs_new = BASE + (won_new * EV).sum(axis=1)
        scores.append((evs_new >= 270).mean())   # chance of reaching 270
    best = np.argmax(scores)      # argmax: where the biggest score is
    plan_270[best] = plan_270[best] + 1

# astype(int): show whole millions
pd.Series(plan_270, index=sw['state']).astype(int)

*Check: PA 34, MI 8, WI 8.*

Now the same loop with one line changed: score each try by **expected electoral votes**, the average across the 200,000 elections, instead of the chance of reaching 270.

In [ ]:
plan_ev = np.zeros(7)

for r in range(50):
    scores = []
    for i in range(7):
        spend = plan_ev.copy()
        spend[i] = spend[i] + 1
        margins = m0 + 40 * coef * (1 - np.exp(-spend / 40))
        won_new = (margins + national + state) > 0
        evs_new = BASE + (won_new * EV).sum(axis=1)
        scores.append(evs_new.mean())   # <- the only line that changed
    best = np.argmax(scores)
    plan_ev[best] = plan_ev[best] + 1

# astype(int): show whole millions
pd.Series(plan_ev, index=sw['state']).astype(int)

*Check: PA 21, GA 13, MI 10, NC 6.*

Score both plans on both goals.

In [ ]:
# same 200,000 elections, each plan's spending added in
margins = m0 + 40 * coef * (1 - np.exp(-plan_ev / 40))
won_new = (margins + national + state) > 0
evs_ev = BASE + (won_new * EV).sum(axis=1)

margins = m0 + 40 * coef * (1 - np.exp(-plan_270 / 40))
won_new = (margins + national + state) > 0
evs_270 = BASE + (won_new * EV).sum(axis=1)

print('expected-EV plan: expected EVs', round(evs_ev.mean(), 1),
      '| chance of 270', round(100 * (evs_ev >= 270).mean(), 1), '%')
print('chance-of-270 plan: expected EVs', round(evs_270.mean(), 1),
      '| chance of 270', round(100 * (evs_270 >= 270).mean(), 1), '%')

*Check: 266.5 and 46.5%, then 266.1 and 47.6%.*

The expected-EV plan wins on expected electoral votes and loses the election more often.

The chance-of-270 plan bought Pennsylvania 19, Michigan 15 and Wisconsin 10. **That is 44, exactly the number you need.** Expected electoral votes counts a vote you did not need the same as one you did, so that plan buys votes in Georgia and North Carolina that mostly arrive in elections you were already winning.

## Part 6: What if the effect is smaller, or zero?

Keep the chance-of-270 plan and change only how much \$1M moves a state: the top of her interval, her estimate, and the bottom.

In [ ]:
for c in [hi, coef, lo]:
    margins = m0 + 40 * c * (1 - np.exp(-plan_270 / 40))
    won_new = (margins + national + state) > 0
    evs_new = BASE + (won_new * EV).sum(axis=1)
    print('if $1M moves a state', round(c, 3), 'points: chance of 270',
          round(100 * (evs_new >= 270).mean(), 1), '%')

# at zero the money moves nothing, so you stay where you started
print('if it moves nothing: chance of 270', round(100 * before, 1), '%')

*Check: 52.0%, 47.6%, 43.2%, 39.5%.*

\$50 million buys somewhere between nothing and 12.5 points of win probability (52.0 − 39.5). The experiments in Kalla and Broockman (2018) cannot rule out zero for ads like these.

---

## Before you leave

1. **Runtime → Restart session and run all.** It runs the notebook from top to bottom, in order. The allocator cells take a few seconds each.
2. **Check that every cell ran.** Scroll from the top. Every code cell should show its output below it.
3. **File → Print → Save as PDF.**
4. **Upload the PDF** to the **in-class** assignment on Canvas.